# 04b. 경량 모델 크기·속도 측정

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 기획서 4장 "왜 경량 모델인가" 표에 넣을 숫자 만들기
- 비교: 경량 모델(로지스틱 회귀) vs 고성능 모델(③ 최신 XGBoost)
- 재는 것: 모델 파일 크기(MB), 환자 1명·500명 예측 시간(ms), 예측 시간 중 전처리와 모델 계산의 비율
- 측정 조건: 같은 컴퓨터(CPU), train 500명 사용 (test 사용 안 함)
- 기록표에는 적지 않음 (결과는 이 노트북의 출력으로 확인)

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [10]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


In [11]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [12]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 2. 측정 준비 — 크기·속도 측정 함수
모델 파일 크기(MB)와 예측 시간(ms)을 같은 컴퓨터에서 재요. 첫 실행은 느려서 한 번 미리 돌린 뒤(워밍업) 여러 번 재서 가운데 값을 써요.

In [13]:
import io, time, joblib
from scipy import sparse
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# 측정 환경 (같은 컴퓨터에서 잰 값끼리만 비교)
print("측정 환경:", platform.platform(), "| python", platform.python_version())

def measure(model, X_sample, n_repeat=20):
    """모델 크기(MB)와 예측 시간(ms). 워밍업 1회 후 반복 측정의 중앙값."""
    buffer = io.BytesIO()
    joblib.dump(model, buffer)                        # 파일로 저장했을 때의 크기
    size_mb = buffer.getbuffer().nbytes / 1024**2

    one = X_sample.iloc[:1]
    model.predict_proba(one)                          # 워밍업
    t_one = []
    for _ in range(n_repeat):
        start = time.perf_counter()
        model.predict_proba(one)
        t_one.append((time.perf_counter() - start) * 1000)

    t_all = []
    for _ in range(5):
        start = time.perf_counter()
        model.predict_proba(X_sample)
        t_all.append((time.perf_counter() - start) * 1000)

    return {"파일 크기(MB)": round(size_mb, 2),
            "1명 예측(ms)": round(float(np.median(t_one)), 2),
            f"{len(X_sample)}명 예측(ms)": round(float(np.median(t_all)), 1)}

X_timing = X.iloc[:500]   # 시간 측정용 샘플 (train에서 가져옴, test는 사용 안 함)
print("측정 함수 준비 완료")

측정 환경: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18
측정 함수 준비 완료


## 3. 경량 모델 — 로지스틱 회귀 E 조합 (② 02_features의 make_features_E와 같은 설정)
0/1 + log(1+변이 개수) + log(1+변이 종류) + log(1+아미노산 쌍) + 유전자별 끊김 표시 (총 9,174열), C=0.1, class_weight=balanced. train 전체로 학습한 뒤 크기·속도를 재요.

In [14]:
from scipy.sparse import csr_matrix
from src.features import (mutation_count, mutation_type_counts,
                          amino_acid_pair_counts, gene_truncation_flags)

def make_features_E(X):
    """② E 조합 피처 (02_features의 make_features_E와 같은 계산). 한 환자 안에서만 계산."""
    return pd.concat([binarize_mutations(X),
                      np.log1p(mutation_count(X)),
                      np.log1p(mutation_type_counts(X)),
                      np.log1p(amino_acid_pair_counts(X)),
                      gene_truncation_flags(X)], axis=1)

def make_features_E_sparse(X):
    return csr_matrix(make_features_E(X).to_numpy(dtype=np.float32))

LR_E_PARAMS = dict(C=0.1, class_weight="balanced", max_iter=3000)   # ② E 실험과 같은 설정

light_model = Pipeline([
    ("features", FunctionTransformer(make_features_E_sparse)),
    ("model", LogisticRegression(**LR_E_PARAMS)),
]).fit(X, y)

light_result = measure(light_model, X_timing)
light_result

{'파일 크기(MB)': 0.94, '1명 예측(ms)': 772.47, '500명 예측(ms)': 3456.5}

## 4. 고성능 모델 — ③ XGBoost E 조합 (04_xgb_feature_E의 xgb_E_group_1109_depth3)
경량 모델과 같은 E 피처, 나무 250개·깊이 3. 측정 조건: 이 컴퓨터의 CPU (서비스 시나리오의 병원 PC 가정)

In [15]:
XGB_E_PARAMS = dict(n_estimators=250, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    objective="multi:softprob", eval_metric="mlogloss",
                    tree_method="hist", random_state=SEED, n_jobs=-1)   # ③ depth3 설정

heavy_model = Pipeline([
    ("features", FunctionTransformer(make_features_E_sparse)),       # 경량 모델과 같은 E 피처
    ("model", XGBClassifier(**XGB_E_PARAMS)),
]).fit(X, y)

heavy_result = measure(heavy_model, X_timing)
heavy_result

{'파일 크기(MB)': 6.78, '1명 예측(ms)': 958.15, '500명 예측(ms)': 3595.6}

## 5. 경량 vs 고성능 비교 (기획서 4장 표)
CV Macro F1은 팀 기록표 값(그룹 K-fold 기준), 크기·속도는 위에서 같은 컴퓨터로 잰 값이에요.

In [16]:
comparison = pd.DataFrame([
    {"모델": "고성능: ③ XGBoost E 조합", "CV Macro F1": 0.4870, **heavy_result},
    {"모델": "경량: 로지스틱 회귀 E 조합", "CV Macro F1": 0.4919, **light_result},
])
print("측정 환경:", platform.platform(), "| python", platform.python_version())
print("크기:", round(heavy_result["파일 크기(MB)"] / light_result["파일 크기(MB)"], 1), "배 차이")
print("1명 예측 속도:", round(heavy_result["1명 예측(ms)"] / light_result["1명 예측(ms)"], 1), "배 차이")
comparison

측정 환경: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18
크기: 7.2 배 차이
1명 예측 속도: 1.2 배 차이


,모델,CV Macro F1,파일 크기(MB),1명 예측(ms),500명 예측(ms)
0,고성능: ③ XGBoost E 조합,0.4870,6.78,958.15,3595.6
1,경량: 로지스틱 회귀 E 조합,0.4919,0.94,772.47,3456.5


## 6. 예측 시간 분해 — 전처리 vs 모델 계산

In [17]:
def measure_split(model, X_sample, n_repeat=20):
    """1명 기준: 전처리 시간과 모델 예측 시간을 나눠 측정 (중앙값, ms)."""
    features, clf = model[:-1], model[-1]
    one = X_sample.iloc[:1]
    one_feat = features.transform(one)
    clf.predict_proba(one_feat)                      # 워밍업
    t_feat, t_model = [], []
    for _ in range(n_repeat):
        start = time.perf_counter()
        features.transform(one)
        t_feat.append((time.perf_counter() - start) * 1000)
        start = time.perf_counter()
        clf.predict_proba(one_feat)
        t_model.append((time.perf_counter() - start) * 1000)
    return {"전처리(ms)": round(float(np.median(t_feat)), 2),
            "모델 예측(ms)": round(float(np.median(t_model)), 3)}

print("경량  :", measure_split(light_model, X_timing))
print("고성능:", measure_split(heavy_model, X_timing))

경량  : {'전처리(ms)': 1099.52, '모델 예측(ms)': 0.205}
고성능: {'전처리(ms)': 1151.04, '모델 예측(ms)': 1.03}


## 7. 앙상블 크기·시간 추정 — ③ XGBoost E + 로지스틱 회귀 E (50:50)

In [18]:
split_light = measure_split(light_model, X_timing)
split_heavy = measure_split(heavy_model, X_timing)

ensemble_result = {
    "파일 크기(MB)": round(light_result["파일 크기(MB)"] + heavy_result["파일 크기(MB)"], 2),
    "1명 예측(ms, 추정)": round(split_light["전처리(ms)"]
                               + split_light["모델 예측(ms)"] + split_heavy["모델 예측(ms)"], 2),
}

final_table = pd.DataFrame([
    {"모델": "앙상블 (XGBoost E + 로지스틱 회귀 E)", "CV Macro F1": 0.5096,
     "파일 크기(MB)": ensemble_result["파일 크기(MB)"], "1명 예측(ms)": ensemble_result["1명 예측(ms, 추정)"]},
    {"모델": "고성능: ③ XGBoost E 조합", "CV Macro F1": 0.4870,
     "파일 크기(MB)": heavy_result["파일 크기(MB)"], "1명 예측(ms)": heavy_result["1명 예측(ms)"]},
    {"모델": "경량: 로지스틱 회귀 E 조합", "CV Macro F1": 0.4919,
     "파일 크기(MB)": light_result["파일 크기(MB)"], "1명 예측(ms)": light_result["1명 예측(ms)"]},
])
print("앙상블 대비 경량 모델 성능 차이:", round(0.5096 - 0.4919, 4), "(기획서 채택 기준 0.02 이하)")
final_table

앙상블 대비 경량 모델 성능 차이: 0.0177 (기획서 채택 기준 0.02 이하)


,모델,CV Macro F1,파일 크기(MB),1명 예측(ms)
0,앙상블 (XGBoost E + 로지스틱 회귀 E),0.5096,7.72,1197.66
1,고성능: ③ XGBoost E 조합,0.4870,6.78,958.15
2,경량: 로지스틱 회귀 E 조합,0.4919,0.94,772.47
